In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# 1) Import Libraries

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# 2) Load the Dataset

In [ ]:
df = pd.read_csv(r'/kaggle/input/weather-dataset-rattle-package/weatherAUS.csv')
df

# 3) Basic Information About Dataset

In [ ]:
df.info()
# Check data types, non-null values, and detect missing data.

In [ ]:
df.describe()
#Generate statistical summary of numerical columns.

# 4) Check Missing Values

In [ ]:
df.isnull().sum()
#Count missing values in each column.

# Cleaninig Data

In [ ]:
df[["Evaporation", "Sunshine"]].isnull().mean() * 100
#Calculate percentage of missing values.

**Evaporation Imputation**

In [ ]:
df["Evaporation"] = df.groupby("Location")["Evaporation"].transform(
    lambda x: x.fillna(x.median())
)
#Fill missing Evaporation values using the median per location.

# Fill using Location Median(Sunshine)

In [ ]:
df["Sunshine"] = df.groupby("Location")["Sunshine"].transform(
    lambda x: x.fillna(x.median())
)

In [ ]:
sns.boxplot(x="Location", y="Evaporation", data=df)
plt.xticks(rotation=90)
plt.show()

In [ ]:
df.isnull().sum()


# First: Basic Digital Columns
Columns:

MinTemp

MaxTemp

Rainfall

WindGustSpeed

WindSpeed9am

WindSpeed3pm

Humidity9am

Humidity3pm

Pressure9am

Pressure3pm

Temp9am

Temp3pm

In [ ]:
numeric_cols = [
    "MinTemp","MaxTemp","Rainfall",
    "WindGustSpeed","WindSpeed9am","WindSpeed3pm",
    "Humidity9am","Humidity3pm",
    "Pressure9am","Pressure3pm",
    "Temp9am","Temp3pm"
]

for col in numeric_cols:
    df[col] = df.groupby("Location")[col].transform(
        lambda x: x.fillna(x.median())
    )
    #Fill missing numerical values using location-based median to preserve geographical weather patterns.

# Second: Wind Direction Columns
Columns:

WindGustDir

WindDir9am

WindDir3pm

Categorical Data

In [ ]:
wind_dir_cols = ["WindGustDir","WindDir9am","WindDir3pm"]

for col in wind_dir_cols:
    df[col] = df.groupby("Location")[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )
    #Fill missing wind direction values using the most frequent direction per location.

# Cloud9am / Cloud3pm

In [ ]:
cloud_cols = ["Cloud9am","Cloud3pm"]

for col in cloud_cols:
    df[col] = df.groupby("Location")[col].transform(
        lambda x: x.fillna(x.median())
    )

    df[col].fillna(df[col].median(), inplace=True)
    #Apply hierarchical imputation (location median then global median) for high-missing cloud variables.

# Pressure Columns

In [ ]:
df

In [ ]:
Pressure = ["Pressure9am","Pressure3pm"]

for col in Pressure :
    df[col] = df.groupby("Location")[col].transform(
        lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
    )
    #Fill missing wind direction values using the most frequent direction per location.

# RainToday  RainTomorrow

In [ ]:
df["RainToday"] = np.where(df["Rainfall"] > 1, "Yes", "No")
# Reconstruct RainToday logically from Rainfall.

In [ ]:
df["Date"] = pd.to_datetime(df["Date"])
df["Month"] = df["Date"].dt.month
#Extract month from Date for seasonal grouping.


# Logical Imputation

In [ ]:
df["WindGustDir"] = df.groupby("Location")["WindGustDir"].transform(
    lambda x: x.fillna(x.mode()[0] if not x.mode().empty else np.nan)
)

In [ ]:
df.isnull().sum()


# Visual analysis
# Analyze RainTomorrow (Target Variable)

In [ ]:
sns.countplot(x="RainTomorrow", data=df)
plt.title("Rain Tomorrow Distribution")
plt.show()
#Visualize the distribution of the target variable.

#  Relationship Between Rainfall and RainTomorrow

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="RainTomorrow", y="Rainfall", data=df)
plt.title("Rainfall vs RainTomorrow")
plt.show()
# Check how rainfall today affects tomorrow's rain.

# Temperature Analysis

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(df["MaxTemp"], bins=30, kde=True)
plt.title("Max Temperature Distribution")
plt.show()
# Visualize the distribution of maximum temperature.

# Correlation Between Numerical Features

In [ ]:
plt.figure(figsize=(12,8))
sns.heatmap(df.corr(numeric_only=True), annot=True, cmap="coolwarm")
plt.title("Correlation Matrix")
plt.show() 
# Identify relationships between numerical variables.

# Humidity Effect on RainTomorrow

In [ ]:
plt.figure(figsize=(6,4))
sns.boxplot(x="RainTomorrow", y="Humidity3pm", data=df)
plt.show()
# Analyze how afternoon humidity affects next day rain.

# Wind Gust Speed Distribution

In [ ]:
plt.figure(figsize=(6,4))
sns.histplot(df["WindGustSpeed"], bins=30, kde=True)
plt.title("Wind Gust Speed Distribution")
plt.show() 
# Understand wind speed behavior.